# Ultimate Tic Tac Toe — AI Battle

**ESILV — Artificial Intelligence Project**

---

This notebook implements an AI agent for the game of Ultimate Tic Tac Toe, designed to compete in the AI Battle. The agent is built around the **Minimax algorithm with Alpha-Beta pruning**, enhanced with iterative deepening, a transposition table, killer-move heuristics, and a custom evaluation function.

## Project constraints

The implementation strictly respects the constraints stated in the project brief:

| Constraint | Status |
|---|---|
| Algorithm based on Minimax with Alpha-Beta pruning | ✅ |
| No precomputed move dictionaries | ✅ |
| Custom heuristic evaluation function | ✅ |
| Depth-limited search (does not explore the full tree) | ✅ |
| Text-mode display, columns and rows numbered 1–9 | ✅ |
| Choice of who starts at the beginning | ✅ |
| Three game modes including external-AI relay | ✅ |
| Compatible with Google Colab | ✅ |

## How to use this notebook

Run the cells in order from top to bottom. The first six cells define the engine, the heuristic, the AI, and the user interface. The final cell launches the game and prompts you to choose a mode.

## Architecture overview

The code is organised in three layers:

1. **Game engine** — represents the board state and enforces the rules of Ultimate Tic Tac Toe (including the *send-rule* that forces the next player into a specific small board).
2. **AI engine** — performs the search and returns the chosen move within a configurable time budget.
3. **Interface** — handles board rendering, coordinate parsing, and the main game loop with three modes (Human vs AI, AI vs AI, AI vs External AI).

Each layer is implemented in its own section below, with comments explaining the key design decisions.

## 1. Imports

The implementation relies only on the Python standard library (`time`, `math`, `random`, `typing`) — no external dependencies are required. This guarantees that the notebook runs on any Colab instance without installation steps.

In [ ]:
import time
import math
import random
from typing import List, Tuple, Optional


## 2. Constants and game representation

### Player encoding

Players are encoded as integers — `X = 1`, `O = -1`, `EMPTY = 0` — so that flipping the side to move is just a sign change (`-player`), and so that summing three cells on a line gives `+3` for an X win, `-3` for an O win, and any mixed value otherwise. This avoids dictionary lookups in the inner search loop and keeps the heuristic simple.

### Lines and incremental evaluation

`LINES` enumerates the 8 winning lines on a 3×3 board (3 rows + 3 columns + 2 diagonals). `CELL_LINES[i]` lists only the lines that pass through cell `i`, allowing future incremental updates without scanning all 8 lines.

### Strategic weights

Both `CELL_WEIGHTS` (within a small board) and `BOARD_WEIGHTS` (within the meta-board) follow the classic Tic Tac Toe principle: **center > corners > edges**. The center cell of a board, and the center board of the meta-board, participate in 4 winning lines each (the most possible), so they are valued highest. Corners participate in 3 lines, edges in only 2.

### Heuristic constants

The four `W_*` constants control the relative weights of the heuristic components — see Section 4 for a detailed discussion. The values were tuned by self-play.

In [ ]:
# =============================================================================
# CONSTANTS
# =============================================================================

EMPTY = 0
X = 1     # Player 1 — crosses (maximizing by convention in human display)
O = -1    # Player 2 — circles
DRAW = 2  # Sentinel for a finished-but-tied small board

# All 8 winning lines on a 3x3 board (cell indices 0..8).
LINES: Tuple[Tuple[int, int, int], ...] = (
    (0, 1, 2), (3, 4, 5), (6, 7, 8),  # rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8),  # columns
    (0, 4, 8), (2, 4, 6),             # diagonals
)

# For each cell, list the winning lines it participates in.
# Used to incrementally evaluate positions without scanning all 8 lines.
CELL_LINES: Tuple[Tuple[Tuple[int, int, int], ...], ...] = tuple(
    tuple(line for line in LINES if i in line) for i in range(9)
)

# Strategic value of each cell within a small 3x3 board.
# Center > corners > edges (matches classic Tic Tac Toe theory).
CELL_WEIGHTS = (3, 2, 3,
                2, 4, 2,
                3, 2, 3)

# Strategic value of each small board within the meta-board.
# Same logic — winning the center board is the most powerful single result.
BOARD_WEIGHTS = (3, 2, 3,
                 2, 4, 2,
                 3, 2, 3)

# Heuristic weights — tuned by self-play.
W_META_LINE      = 30.0   # Threats on the meta-board (most important)
W_META_OWNED     = 6.0    # Bonus for owning a strategically valuable board
W_LOCAL_LINE     = 1.0    # Threats inside a single small board
W_POSITIONAL     = 0.3    # Bonus for owning strong cells inside a board
W_FREE_PENALTY   = 0.0    # Reserved (handled in move ordering instead)

# Score bounds — safely outside any heuristic value.
WIN_SCORE  = 1_000_000
INF        = float('inf')



## 3. The game engine — `UltimateBoard`

This class holds the full state of a game and exposes the rules of Ultimate Tic Tac Toe.

### State representation

- `small_boards[b][c]` — value of cell `c` in small board `b` (X / O / EMPTY).
- `meta_board[b]` — winner of small board `b` (X / O / DRAW / EMPTY).
- `forced_board` — the small board where the next move must land, or `-1` for free choice.
- `current_player` — whose turn it is (X or O).
- `history` — LIFO stack of move records that supports `undo_move`.

### Why no copying

A single Minimax call may visit tens of thousands of nodes. Copying the full board state at every step would be prohibitively slow in pure Python. Instead, the engine **mutates a single shared instance** and reverses the mutation in last-in-first-out order via `undo_move`. Every move stored on the history stack records all the information needed to reverse it perfectly, including the previous Zobrist hash.

### Zobrist hashing

Each `(board, cell, player)` triple is associated with a random 64-bit integer at class-load time. The hash of a position is the XOR of all the random values for the marks currently on the board, plus extras for the `forced_board` and the side to move. Updates are incremental: each `make_move` toggles 3 XORs, each `undo_move` reverts them. This hash is the key of the transposition table used by the AI (Section 5).

### The send-rule

After a move at cell `c` in small board `b`, the *cell index* `c` becomes the next forced board — unless small board `c` is already won or drawn, in which case the opponent has free choice. This is the rule that makes Ultimate Tic Tac Toe deeper than its parent: every move is simultaneously a tactical decision (where to mark) and a strategic one (where to send the opponent).

In [ ]:
# =============================================================================
# GAME ENGINE
# =============================================================================

class UltimateBoard:
    """
    Full state of an Ultimate Tic Tac Toe game.

    Layout:
        small_boards[b][c]  : value (EMPTY/X/O) of cell c in small board b
        meta_board[b]       : winner of small board b (EMPTY/X/O/DRAW)
        forced_board        : the small board where the next move MUST land
                              (-1 means "free choice" — first move, or the
                               target board is already finished)
        current_player      : X or O — whose turn it is
        history             : LIFO stack of (board, cell, prev_forced,
                              prev_meta, prev_zobrist) to support undo

    All search code uses make_move/undo_move on a single shared instance —
    we never copy the state, which is a major speed win.
    """

    __slots__ = ("small_boards", "meta_board", "forced_board",
                 "current_player", "history", "zobrist")

    # Zobrist hash table — one random 64-bit number per (board, cell, player)
    # plus extras for the forced-board and side-to-move.
    # Generated once at class-load time so all instances share the same hash.
    _ZOBRIST_CELLS = [[[random.getrandbits(64) for _ in range(2)]
                       for _ in range(9)] for _ in range(9)]
    _ZOBRIST_FORCED = [random.getrandbits(64) for _ in range(10)]  # -1..8 -> 0..9
    _ZOBRIST_SIDE = random.getrandbits(64)

    def __init__(self):
        self.small_boards: List[List[int]] = [[EMPTY] * 9 for _ in range(9)]
        self.meta_board: List[int] = [EMPTY] * 9
        self.forced_board: int = -1
        self.current_player: int = X
        self.history: List[Tuple[int, int, int, int, int]] = []
        self.zobrist: int = self._ZOBRIST_FORCED[0]  # forced_board = -1

    # -------------------------------------------------------------------------
    # Move generation
    # -------------------------------------------------------------------------

    def legal_moves(self) -> List[Tuple[int, int]]:
        """Return all legal (board_index, cell_index) tuples."""
        moves: List[Tuple[int, int]] = []
        if self.forced_board != -1 and self.meta_board[self.forced_board] == EMPTY:
            b = self.forced_board
            cells = self.small_boards[b]
            for c in range(9):
                if cells[c] == EMPTY:
                    moves.append((b, c))
        else:
            for b in range(9):
                if self.meta_board[b] == EMPTY:
                    cells = self.small_boards[b]
                    for c in range(9):
                        if cells[c] == EMPTY:
                            moves.append((b, c))
        return moves

    # -------------------------------------------------------------------------
    # Mutation (with full undo support)
    # -------------------------------------------------------------------------

    def make_move(self, board: int, cell: int) -> None:
        """Apply a move. State changes are reversible via undo_move()."""
        prev_forced = self.forced_board
        prev_meta = self.meta_board[board]
        prev_zobrist = self.zobrist

        player = self.current_player
        self.small_boards[board][cell] = player

        # Zobrist: toggle the cell hash (player index 0 for X, 1 for O)
        p_idx = 0 if player == X else 1
        self.zobrist ^= self._ZOBRIST_CELLS[board][cell][p_idx]

        # Did this complete the small board?
        new_meta = self._small_board_status(board)
        self.meta_board[board] = new_meta

        # Determine next forced board.
        # Rule: the cell index of the move dictates which board the opponent
        # must play in. If that target board is already finished, free choice.
        if self.meta_board[cell] != EMPTY:
            new_forced = -1
        else:
            new_forced = cell

        # Zobrist: update forced-board hash and side-to-move
        self.zobrist ^= self._ZOBRIST_FORCED[prev_forced + 1]
        self.zobrist ^= self._ZOBRIST_FORCED[new_forced + 1]
        self.zobrist ^= self._ZOBRIST_SIDE

        self.forced_board = new_forced
        self.history.append((board, cell, prev_forced, prev_meta, prev_zobrist))
        self.current_player = -player

    def undo_move(self) -> None:
        """Reverse the last move. Must be called in LIFO order."""
        board, cell, prev_forced, prev_meta, prev_zobrist = self.history.pop()
        self.current_player = -self.current_player
        self.small_boards[board][cell] = EMPTY
        self.meta_board[board] = prev_meta
        self.forced_board = prev_forced
        self.zobrist = prev_zobrist

    # -------------------------------------------------------------------------
    # Status queries
    # -------------------------------------------------------------------------

    def _small_board_status(self, b: int) -> int:
        """Return X / O if won, DRAW if full and tied, EMPTY otherwise."""
        cells = self.small_boards[b]
        for (i, j, k) in LINES:
            s = cells[i] + cells[j] + cells[k]
            if s == 3:
                return X
            if s == -3:
                return O
        for v in cells:
            if v == EMPTY:
                return EMPTY
        return DRAW

    def winner(self) -> int:
        """Return X / O if someone has won the global game, else EMPTY."""
        m = self.meta_board
        for (i, j, k) in LINES:
            a, b, c = m[i], m[j], m[k]
            if a == b == c and (a == X or a == O):
                return a
        return EMPTY

    def is_terminal(self) -> bool:
        """True if the game is over (winner declared or no legal moves left)."""
        if self.winner() != EMPTY:
            return True
        return len(self.legal_moves()) == 0

    def board_count(self, player: int) -> int:
        """Number of small boards won by `player` — used as draw tiebreaker."""
        return sum(1 for v in self.meta_board if v == player)



## 4. The heuristic evaluation function

When the search reaches its depth limit at a non-terminal position, the AI must **estimate** how favourable the position is. The quality of this estimate almost entirely determines the strength of the AI: a perfect search with a poor heuristic plays badly, and a strong heuristic with a shallow search plays well.

The heuristic combines four signals, ordered from most to least important:

### Component A — Meta-board threats (dominant)

The real goal of the game is to win three small boards in a row, **not** to win individual cells. The dominant term therefore evaluates the meta-board exactly as if it were a regular Tic Tac Toe board. For each of the 8 meta-lines, we count how many small boards each side owns and assign:

- 2-in-a-row + 1 empty board → strong threat (+10 × `W_META_LINE`)
- 1-in-a-row + 2 empty boards → potential (+1 × `W_META_LINE`)
- Contested line (both players have at least one) → 0 (the line is dead)
- Mirror values for the opponent.

`W_META_LINE = 30` makes this term roughly an order of magnitude larger than any other component, which is what makes the AI play *strategically* rather than greedily.

### Component B — Strategic ownership of small boards

Owning a small board is rewarded directly by `W_META_OWNED × board weight`. This complements component A — even when no immediate meta-threat exists, valuable boards are worth acquiring because they will participate in future threats. This term also handles the AI Battle's tie-break rule (most-boards-won wins drawn games) without a separate clause.

### Component C — Per-small-board control

For each unfinished small board, the same line-scoring logic is applied to its 3×3 grid of cells, then multiplied by the strategic weight of that board. Fighting in the center board is more valuable than fighting in a corner board, which is more valuable than fighting in an edge board.

### Component D — Positional bonus

A small bonus is added for occupying central or corner cells inside a small board. This is the only purely positional term — every other component is line-based.

### The blocked-line insight

A line containing marks of both players is dead — neither side can ever complete it. Returning **0** for these lines (instead of counting individual marks) is a small change with a large effect. Without it, the heuristic would over-value positions where the AI has scattered marks on contested lines, leading to greedy, unfocused play. This is a common pitfall in naive Tic Tac Toe heuristics.

In [ ]:
# =============================================================================
# HEURISTIC EVALUATION
# =============================================================================

def _line_score(a: int, b: int, c: int, player: int) -> int:
    """
    Score a single 3-cell line from `player`'s perspective.

    Reasoning:
      * A line with marks of both players is dead — score 0.
      * 3-in-a-row is a win on this line — return a large value.
      * 2-in-a-row with one empty cell = immediate threat — high reward.
      * 1-in-a-row with two empties = potential — small reward.
      * Mirror values for the opponent.
    """
    p_count = (a == player) + (b == player) + (c == player)
    o_count = (a == -player) + (b == -player) + (c == -player)

    if p_count and o_count:
        return 0  # Blocked line — no future value to either side
    if p_count == 3:
        return 100
    if p_count == 2:
        return 10
    if p_count == 1:
        return 1
    if o_count == 3:
        return -100
    if o_count == 2:
        return -10
    if o_count == 1:
        return -1
    return 0


def _evaluate_small_board(cells: List[int], player: int) -> float:
    """Sum of line scores plus a small positional bonus for occupied cells."""
    score = 0.0
    for (i, j, k) in LINES:
        score += _line_score(cells[i], cells[j], cells[k], player)
    for idx in range(9):
        v = cells[idx]
        if v == player:
            score += CELL_WEIGHTS[idx] * W_POSITIONAL
        elif v == -player:
            score -= CELL_WEIGHTS[idx] * W_POSITIONAL
    return score


def evaluate(state: UltimateBoard, player: int) -> float:
    """
    Full heuristic from `player`'s perspective.

    Combines:
      (1) Has the global game been won?       — terminal short-circuit
      (2) Meta-board threats                  — DOMINANT term (W_META_LINE)
      (3) Strategic ownership of small boards — bonus (W_META_OWNED)
      (4) Per-small-board control             — fine-grained tactical signal
    """
    # (1) Terminal check (cheap — caller usually does this too, but safe)
    w = state.winner()
    if w == player:
        return WIN_SCORE
    if w == -player:
        return -WIN_SCORE

    score = 0.0

    # (2) Meta-board: threats on the 3x3 of board-winners.
    # DRAW counts as "blocked" for both sides, so map it to a neutral marker.
    meta = state.meta_board
    m0 = meta[0] if meta[0] in (X, O) else EMPTY
    m1 = meta[1] if meta[1] in (X, O) else EMPTY
    m2 = meta[2] if meta[2] in (X, O) else EMPTY
    m3 = meta[3] if meta[3] in (X, O) else EMPTY
    m4 = meta[4] if meta[4] in (X, O) else EMPTY
    m5 = meta[5] if meta[5] in (X, O) else EMPTY
    m6 = meta[6] if meta[6] in (X, O) else EMPTY
    m7 = meta[7] if meta[7] in (X, O) else EMPTY
    m8 = meta[8] if meta[8] in (X, O) else EMPTY
    # Treat drawn small boards as walls — they can't contribute to either side.
    # We model this by giving the line-scorer a neutral value if either is DRAW.
    meta_clean = (m0, m1, m2, m3, m4, m5, m6, m7, m8)

    meta_line_score = 0
    for (i, j, k) in LINES:
        a, b, c = meta_clean[i], meta_clean[j], meta_clean[k]
        # If any cell on this meta-line is a drawn board, the line is dead.
        if meta[i] == DRAW or meta[j] == DRAW or meta[k] == DRAW:
            continue
        meta_line_score += _line_score(a, b, c, player)
    score += meta_line_score * W_META_LINE

    # (3) Strategic ownership bonus
    for b in range(9):
        if meta[b] == player:
            score += BOARD_WEIGHTS[b] * W_META_OWNED
        elif meta[b] == -player:
            score -= BOARD_WEIGHTS[b] * W_META_OWNED

    # (4) Per-small-board tactical evaluation (only unfinished boards)
    for b in range(9):
        if meta[b] == EMPTY:
            local = _evaluate_small_board(state.small_boards[b], player)
            # Multiply by board's strategic weight — fighting in the center
            # board matters more than fighting in a corner board.
            score += local * (BOARD_WEIGHTS[b] / 4.0) * W_LOCAL_LINE

    return score



## 5. The AI — Minimax with Alpha-Beta pruning

This section contains the search algorithm, which is the core of the project.

### 5.1 Minimax and Alpha-Beta pruning

**Minimax** assumes both players play optimally: at the AI's nodes, the algorithm picks the move that maximizes the value; at the opponent's nodes, the move that minimizes it. **Alpha-Beta pruning** maintains a window `[α, β]` of values still relevant — when a child's value falls outside this window, the remaining children can be ignored without affecting the final result. With *perfect* move ordering this reduces complexity from O(b^d) to O(b^(d/2)), which roughly **doubles** the depth searchable in a fixed time budget.

The implementation uses the **negamax** form of Minimax. Negamax is mathematically equivalent to Minimax for zero-sum games but expresses both players' logic as a single "maximize from your own perspective" rule, with scores negated when returning to the parent. This avoids duplicating max/min branches.

### 5.2 Iterative deepening

Rather than running a single fixed-depth search, the AI runs successive searches at depths 1, 2, 3, … up to the configured maximum. Two practical advantages:

1. **Anytime behaviour** — if the time budget runs out mid-search, the AI returns the best move from the deepest *fully completed* iteration. There are no wasted calls.
2. **Better move ordering** — the result of one iteration is stored in the transposition table and used as the first move to try at the next iteration. Since alpha-beta is most effective when the best move is searched first, this dramatically improves cutoff rates and more than compensates for the cost of the shallower searches.

### 5.3 Transposition table

Repeated positions occur frequently because moves can be transposed. A hash table keyed on the Zobrist hash records, for every visited position, its search depth, the score returned, a flag (exact, lower bound, or upper bound), and the best move found. When the same position is reached again at equal or greater depth, the stored score short-circuits the search; even when it cannot, the stored best move provides an excellent move-ordering hint for the current iteration.

### 5.4 Killer-move heuristic

When a move causes a beta cutoff, it is recorded as a *killer* for the current ply. At sibling nodes at the same ply, killer moves are tried early during move ordering. This is a cheap and effective way to improve cutoff rates: a move that refutes the opponent at one branch often refutes them at a sibling branch.

### 5.5 Move ordering

Move ordering is the **single most important** optimization for alpha-beta. The AI scores each candidate move with a fast static evaluator and tries the highest-scoring moves first. The score combines, in order of priority:

1. The best move from the transposition table (huge bonus, +100 000)
2. Killer moves for the current ply (+5 000 / +2 500)
3. Moves that immediately complete a 3-in-a-row in the local board (+800)
4. Moves that block an opponent's 3-in-a-row (+400)
5. Moves that create a 2-in-a-row threat (+30)
6. Strategic positional bonuses (cell weight + board weight)
7. A penalty for sending the opponent to a finished board (= free move for them, −25)

### 5.6 Time management and safe state restoration

A deadline is computed at the start of each move. The elapsed time is checked roughly every 1024 nodes using a cheap bitmask test (`nodes_visited & 1023 == 0`). When the deadline is exceeded, a `TimeoutError` is raised. To guarantee that the externally-visible state is always restored to its starting position when `choose_move` returns — even if the timeout fires deep inside a sequence of `make_move` calls — the search uses two layers of defence:

- **Snapshot/restore at the boundary**: `choose_move` saves the public state at entry and restores it unconditionally on exit, in a `finally` block.
- **`try`/`finally` around every move**: each `make_move` is paired with `undo_move` in a `try`/`finally` block, so unwinding the stack is also clean.

### 5.7 Depth-aware terminal scoring

Wins return `WIN_SCORE − (64 − depth)` and losses return `−WIN_SCORE + (64 − depth)`. The depth term causes the AI to **prefer faster wins and slower losses**: between two winning continuations, the shorter is selected; between two losing continuations, the longer is selected (giving the opponent more chances to make a mistake). Without this term, the AI would consider all winning continuations equivalent and could in principle stall.

In [ ]:
# =============================================================================
# AI: ITERATIVE-DEEPENING NEGAMAX WITH ALPHA-BETA PRUNING
# =============================================================================

# Transposition-table flags
TT_EXACT = 0
TT_LOWER = 1  # alpha cutoff (failed-low: real value >= score)
TT_UPPER = 2  # beta  cutoff (failed-high: real value <= score)


class AlphaBetaAI:
    """
    Iterative-deepening alpha-beta search with several enhancements.

    Parameters
    ----------
    player : int
        X or O — the side this AI plays.
    max_depth : int
        Hard ceiling on search depth (per iteration of iterative deepening).
    time_limit : float
        Seconds per move. Soft limit — we abort the current iteration if
        it exceeds the budget, but keep the result of the deepest completed
        iteration.

    Enhancements
    ------------
    * Iterative deepening with anytime move return.
    * Transposition table keyed on Zobrist hash (with depth + flag).
    * Killer moves — store one move per ply that caused a cutoff,
      try it first in sibling nodes.
    * Move ordering combining: TT best move > immediate small-board win >
      killer move > positional static score.
    * Depth-aware terminal scores — prefer fast wins / slow losses.
    """

    def __init__(self, player: int, max_depth: int = 8, time_limit: float = 4.0):
        self.player = player
        self.max_depth = max_depth
        self.time_limit = time_limit

        # Per-search state
        self.start_time = 0.0
        self.deadline = 0.0
        self.nodes_visited = 0
        self.last_completed_depth = 0
        self.tt: dict = {}                                  # zobrist -> (depth, flag, score, best_move)
        self.killers: List[List[Optional[Tuple[int, int]]]] = [[None, None] for _ in range(64)]

    # -------------------------------------------------------------------------
    # Public API
    # -------------------------------------------------------------------------

    def choose_move(self, state: UltimateBoard) -> Optional[Tuple[int, int]]:
        """Return the best move found within the time budget.

        IMPORTANT: a TimeoutError can be raised from anywhere inside the
        recursive search, even mid-way through a sequence of make_move /
        undo_move calls. We MUST guarantee that the externally-visible
        state is identical when this method returns to what it was when
        it was called — otherwise phantom moves leak onto the real board.

        We do this with two layers of defence:
          1) Snapshot the public state at entry and restore it before
             returning (cheap — a few list copies, not deep copies).
          2) Inside the search, every make_move is paired with undo_move
             via try/finally so unwinding the stack is also clean.
        """
        self.start_time = time.time()
        self.deadline = self.start_time + self.time_limit
        self.nodes_visited = 0
        self.last_completed_depth = 0
        self.tt.clear()
        self.killers = [[None, None] for _ in range(64)]

        legal = state.legal_moves()
        if not legal:
            return None
        if len(legal) == 1:
            return legal[0]

        # --- Snapshot the externally-visible state ------------------------
        # We rebuild from this on exit, no matter how the search exits.
        snapshot_small = [list(b) for b in state.small_boards]
        snapshot_meta = list(state.meta_board)
        snapshot_forced = state.forced_board
        snapshot_player = state.current_player
        snapshot_zobrist = state.zobrist
        snapshot_history_len = len(state.history)

        best_move = legal[0]
        best_score = -INF

        try:
            # Iterative deepening
            for depth in range(1, self.max_depth + 1):
                try:
                    score, move = self._root_search(state, depth)
                except TimeoutError:
                    break  # discard partial results from this depth
                best_move, best_score = move, score
                self.last_completed_depth = depth

                # Stop early on a guaranteed mate
                if best_score >= WIN_SCORE - 1000:
                    break
        finally:
            # --- Restore the snapshot UNCONDITIONALLY --------------------
            # If anything inside the search left the state mutated (e.g.
            # because of timeout-during-make_move), this guarantees the
            # caller sees a pristine board.
            for i in range(9):
                state.small_boards[i] = snapshot_small[i]
            state.meta_board = snapshot_meta
            state.forced_board = snapshot_forced
            state.current_player = snapshot_player
            state.zobrist = snapshot_zobrist
            # Truncate history back to its starting length
            del state.history[snapshot_history_len:]

        return best_move

    # -------------------------------------------------------------------------
    # Root search — like a normal node, but tracks the best move
    # -------------------------------------------------------------------------

    def _root_search(self, state: UltimateBoard,
                     depth: int) -> Tuple[float, Tuple[int, int]]:
        alpha, beta = -INF, INF
        legal = state.legal_moves()

        # Try the previous iteration's best move first — strong ordering signal.
        tt_entry = self.tt.get(state.zobrist)
        tt_best = tt_entry[3] if tt_entry else None
        ordered = self._order_moves(state, legal, tt_best, ply=0)

        best_score = -INF
        best_move = ordered[0]

        for move in ordered:
            state.make_move(*move)
            try:
                score = -self._negamax(state, depth - 1, -beta, -alpha, ply=1)
            finally:
                state.undo_move()

            if score > best_score:
                best_score = score
                best_move = move
            if score > alpha:
                alpha = score
            # No beta cutoff at the root (alpha == -INF initially), but kept
            # for symmetry if we ever pre-seed alpha.

        return best_score, best_move

    # -------------------------------------------------------------------------
    # Recursive negamax
    # -------------------------------------------------------------------------

    def _negamax(self, state: UltimateBoard, depth: int,
                 alpha: float, beta: float, ply: int) -> float:
        # Periodic time check (every ~1024 nodes) — cheap bitmask test
        self.nodes_visited += 1
        if (self.nodes_visited & 1023) == 0 and time.time() > self.deadline:
            raise TimeoutError()

        alpha_orig = alpha

        # --- Transposition table probe -------------------------------------
        tt_entry = self.tt.get(state.zobrist)
        tt_best = None
        if tt_entry is not None:
            tt_depth, tt_flag, tt_score, tt_best = tt_entry
            if tt_depth >= depth:
                if tt_flag == TT_EXACT:
                    return tt_score
                if tt_flag == TT_LOWER and tt_score > alpha:
                    alpha = tt_score
                elif tt_flag == TT_UPPER and tt_score < beta:
                    beta = tt_score
                if alpha >= beta:
                    return tt_score

        # --- Terminal / leaf -----------------------------------------------
        w = state.winner()
        if w != EMPTY:
            # Depth-aware: the closer the win, the better; the closer the
            # loss, the worse. Prevents the AI from stalling out winning lines.
            if w == state.current_player:
                return WIN_SCORE - (64 - depth)
            else:
                return -WIN_SCORE + (64 - depth)

        if depth <= 0:
            return evaluate(state, state.current_player)

        legal = state.legal_moves()
        if not legal:
            # Drawn position — break tie by board count.
            mine = state.board_count(state.current_player)
            theirs = state.board_count(-state.current_player)
            return (mine - theirs) * 50

        # --- Search children -----------------------------------------------
        ordered = self._order_moves(state, legal, tt_best, ply)

        best_score = -INF
        best_move = ordered[0]

        for move in ordered:
            state.make_move(*move)
            try:
                score = -self._negamax(state, depth - 1, -beta, -alpha, ply + 1)
            finally:
                state.undo_move()

            if score > best_score:
                best_score = score
                best_move = move
            if score > alpha:
                alpha = score
            if alpha >= beta:
                # Beta cutoff — record killer move and stop.
                if ply < len(self.killers):
                    k0, k1 = self.killers[ply]
                    if move != k0:
                        self.killers[ply][1] = k0
                        self.killers[ply][0] = move
                break

        # --- Store in transposition table ----------------------------------
        if best_score <= alpha_orig:
            flag = TT_UPPER
        elif best_score >= beta:
            flag = TT_LOWER
        else:
            flag = TT_EXACT
        self.tt[state.zobrist] = (depth, flag, best_score, best_move)

        return best_score

    # -------------------------------------------------------------------------
    # Move ordering — critical for alpha-beta efficiency
    # -------------------------------------------------------------------------

    def _order_moves(self, state: UltimateBoard,
                     moves: List[Tuple[int, int]],
                     tt_best: Optional[Tuple[int, int]],
                     ply: int) -> List[Tuple[int, int]]:
        """
        Sort moves best-first using a fast static estimate. Good ordering
        can take alpha-beta from O(b^d) toward O(b^(d/2)), so this matters
        more than almost any other tweak.
        """
        killer0 = self.killers[ply][0] if ply < len(self.killers) else None
        killer1 = self.killers[ply][1] if ply < len(self.killers) else None
        player = state.current_player
        meta = state.meta_board
        small = state.small_boards

        scored = []
        for mv in moves:
            b, c = mv
            score = 0

            # Highest priority: the move recommended by the transposition table
            if tt_best is not None and mv == tt_best:
                score += 100_000

            # Killer moves (cutoffs in sibling nodes at the same ply)
            if mv == killer0:
                score += 5_000
            elif mv == killer1:
                score += 2_500

            # Quick: would this move win the small board?
            cells = small[b]
            for (i, j, k) in CELL_LINES[c]:
                # Sum of the OTHER two cells on the line — if it equals 2*player,
                # placing here completes 3-in-a-row.
                others = (cells[i] if i != c else 0) + \
                         (cells[j] if j != c else 0) + \
                         (cells[k] if k != c else 0)
                if others == 2 * player:
                    score += 800   # winning move on this line
                elif others == 2 * -player:
                    score += 400   # blocking opponent's 3-in-a-row
                elif others == player:
                    score += 30    # creating a 2-in-a-row threat

            # Bonus for completing a small board that's strategically important
            score += BOARD_WEIGHTS[b] * 4

            # Static positional value of the cell
            score += CELL_WEIGHTS[c]

            # PENALTY: sending opponent to a finished board (= free move).
            # We don't want to give the opponent a free choice unless it's
            # tactically justified.
            if meta[c] != EMPTY:
                score -= 25

            # BONUS: sending opponent to a board where they have few good moves.
            # Cheap proxy: count their existing presence in board `c`.
            else:
                opp_in_target = sum(1 for v in small[c] if v == -player)
                my_in_target = sum(1 for v in small[c] if v == player)
                score += (my_in_target - opp_in_target) * 2

            scored.append((-score, mv))

        scored.sort(key=lambda t: t[0])
        return [mv for _, mv in scored]



## 6. Text interface — rendering and coordinate parsing

The interface is text-based, as authorised by the project specification. It is intentionally compact and unambiguous:

- The 9×9 grid is printed with vertical and horizontal separators that delimit the nine small boards.
- The meta-board is shown beneath it (which small boards are won, by whom, or drawn).
- The next forced board (or *FREE choice*) is printed below.

Coordinates follow the project specification: **columns and rows are numbered 1 to 9, the column is entered first**, separated by a space. For example, `5 5` plays in the dead center cell of the dead center board.

The two helper functions `coords_to_move` and `move_to_coords` translate between user-facing `(column, row)` pairs and the internal `(board_index, cell_index)` representation used by the engine.

In [ ]:
# =============================================================================
# TEXT INTERFACE
# =============================================================================

def render(state: UltimateBoard) -> None:
    """Pretty-print the 9x9 board with separators between small boards."""
    sym = {EMPTY: '.', X: 'X', O: 'O'}
    print()
    print("      1 2 3   4 5 6   7 8 9   <- col")
    print("    +-------+-------+-------+")
    for row in range(9):
        line = f"  {row + 1} | "
        for col in range(9):
            big_row, small_row = row // 3, row % 3
            big_col, small_col = col // 3, col % 3
            board_idx = big_row * 3 + big_col
            cell_idx = small_row * 3 + small_col
            line += sym[state.small_boards[board_idx][cell_idx]] + " "
            if col % 3 == 2:
                line += "| "
        print(line)
        if row % 3 == 2:
            print("    +-------+-------+-------+")
    print("    ^ row")

    meta_sym = {EMPTY: '.', X: 'X', O: 'O', DRAW: '='}
    print("\n  Meta-board (won small boards):")
    for r in range(3):
        print("      " + " ".join(meta_sym[state.meta_board[r * 3 + c]]
                                  for c in range(3)))

    if state.forced_board != -1:
        fb = state.forced_board
        print(f"\n  Next move must land in small board #{fb + 1} "
              f"(big-row {fb // 3 + 1}, big-col {fb % 3 + 1})")
    else:
        print("\n  Next move: FREE choice (any unfinished board)")


def coords_to_move(col: int, row: int) -> Tuple[int, int]:
    """Convert 1-indexed (col, row) UI coordinates into (board_idx, cell_idx)."""
    c = col - 1
    r = row - 1
    big_row, small_row = r // 3, r % 3
    big_col, small_col = c // 3, c % 3
    return big_row * 3 + big_col, small_row * 3 + small_col


def move_to_coords(board: int, cell: int) -> Tuple[int, int]:
    """Inverse of coords_to_move — for displaying AI moves."""
    big_row, big_col = board // 3, board % 3
    small_row, small_col = cell // 3, cell % 3
    r = big_row * 3 + small_row
    c = big_col * 3 + small_col
    return c + 1, r + 1


## 7. The game controller — `Game`

The top-level controller drives the main game loop and offers three modes:

### Mode 1 — Human vs AI

The user chooses who starts. The AI's move is announced in detail (column, row, depth reached, nodes visited, time elapsed) so the user can verify that the AI is searching meaningfully.

### Mode 2 — AI vs AI (autonomous)

Two AI instances play autonomously. This is useful for self-play testing, performance measurement, and tuning the heuristic weights.

### Mode 3 — AI vs External AI

In this mode the user relays the opponent AI's moves manually. **This is the mode used during the inter-group battle**, where two AIs run on different machines and humans act as intermediaries to enter each AI's chosen move into the opposing instance.

All three modes use the same engine, the same AI, and the same display — only the source of moves differs.

In [ ]:
# =============================================================================
# GAME LOOP
# =============================================================================

class Game:
    """Top-level controller: handles mode selection and the main game loop."""

    def __init__(self, ai_depth: int = 8, ai_time: float = 4.0):
        self.ai_depth = ai_depth
        self.ai_time = ai_time

    def run(self) -> None:
        print("=" * 60)
        print("  ULTIMATE TIC TAC TOE — AI BATTLE")
        print("=" * 60)
        print("  1) Human vs AI")
        print("  2) AI vs AI (autonomous)")
        print("  3) AI vs External AI (you relay opponent moves)")
        choice = input("Mode? [1/2/3] (default 1): ").strip() or "1"

        if choice == "1":
            self._human_vs_ai()
        elif choice == "2":
            self._ai_vs_ai()
        else:
            self._ai_vs_external()

    # -------------------------------------------------------------------------

    def _human_vs_ai(self) -> None:
        first = input("Who starts? [h]uman / [a]i (default h): ").strip().lower() or "h"
        human_plays = X if first == "h" else O
        ai = AlphaBetaAI(player=-human_plays,
                         max_depth=self.ai_depth,
                         time_limit=self.ai_time)

        state = UltimateBoard()
        while not state.is_terminal():
            render(state)
            if state.current_player == human_plays:
                move = self._read_human_move(state)
                col, row = move_to_coords(*move)
                print(f"  > You played: column {col}, row {row}")
            else:
                print(f"\n  AI is thinking (depth<={self.ai_depth}, "
                      f"<={self.ai_time}s)...")
                t0 = time.time()
                move = ai.choose_move(state)
                dt = time.time() - t0
                col, row = move_to_coords(*move)
                print(f"  > AI plays: column {col}, row {row}    "
                      f"[depth {ai.last_completed_depth}, "
                      f"{ai.nodes_visited} nodes, {dt:.2f}s]")
            state.make_move(*move)

        render(state)
        self._announce_result(state)

    def _ai_vs_ai(self) -> None:
        ai_x = AlphaBetaAI(X, self.ai_depth, self.ai_time)
        ai_o = AlphaBetaAI(O, self.ai_depth, self.ai_time)
        state = UltimateBoard()
        move_num = 0
        while not state.is_terminal():
            move_num += 1
            ai = ai_x if state.current_player == X else ai_o
            symbol = "X" if state.current_player == X else "O"
            t0 = time.time()
            move = ai.choose_move(state)
            dt = time.time() - t0
            col, row = move_to_coords(*move)
            print(f"Move {move_num:3d} [{symbol}]: col={col}, row={row}    "
                  f"[depth {ai.last_completed_depth}, "
                  f"{ai.nodes_visited} nodes, {dt:.2f}s]")
            state.make_move(*move)
        render(state)
        self._announce_result(state)

    def _ai_vs_external(self) -> None:
        first = input("Who starts? [m]y AI / [o]pponent (default m): ").strip().lower() or "m"
        my_player = X if first == "m" else O
        ai = AlphaBetaAI(my_player, self.ai_depth, self.ai_time)

        state = UltimateBoard()
        while not state.is_terminal():
            render(state)
            if state.current_player == my_player:
                print("\n  My AI is thinking...")
                t0 = time.time()
                move = ai.choose_move(state)
                dt = time.time() - t0
                col, row = move_to_coords(*move)
                print(f"\n  >>> My AI plays: column {col}, row {row} <<<    "
                      f"[depth {ai.last_completed_depth}, "
                      f"{ai.nodes_visited} nodes, {dt:.2f}s]")
            else:
                print("\n  Enter the opponent AI's move:")
                move = self._read_human_move(state)
            state.make_move(*move)

        render(state)
        self._announce_result(state)

    # -------------------------------------------------------------------------

    def _read_human_move(self, state: UltimateBoard) -> Tuple[int, int]:
        legal = set(state.legal_moves())
        while True:
            try:
                raw = input("  Your move (column row, e.g. '5 5'): ").strip()
                parts = raw.split()
                col, row = int(parts[0]), int(parts[1])
                if not (1 <= col <= 9 and 1 <= row <= 9):
                    print("  ! Coordinates must be between 1 and 9.")
                    continue
                move = coords_to_move(col, row)
                if move not in legal:
                    print("  ! Illegal move (cell occupied or wrong board).")
                    continue
                return move
            except (ValueError, IndexError):
                print("  ! Please enter two numbers separated by a space.")

    def _announce_result(self, state: UltimateBoard) -> None:
        print("\n" + "=" * 60)
        w = state.winner()
        if w == X:
            print("  RESULT: X wins!")
        elif w == O:
            print("  RESULT: O wins!")
        else:
            xb = state.board_count(X)
            ob = state.board_count(O)
            print(f"  RESULT: Draw on alignment. "
                  f"X owns {xb} small boards, O owns {ob}.")
            if xb > ob:
                print("  Tiebreaker: X wins on board count.")
            elif ob > xb:
                print("  Tiebreaker: O wins on board count.")
            else:
                print("  True draw.")
        print("=" * 60)



## 8. Start the game

Run the cell below to launch.

### Tuning the speed/strength trade-off

Two parameters control the trade-off between thinking time and playing strength:

- **`ai_depth`** — hard ceiling on search depth. Iterative deepening usually stops earlier when the time budget runs out, so increasing this rarely hurts. Default: `8`.
- **`ai_time`** — soft per-move budget in seconds. The current iteration finishes; deeper iterations are aborted if the deadline passes. Default: `4.0`.

**Adjust `ai_time` to match the per-move clock you will be given for the battle.** For a fast clock try `Game(ai_depth=6, ai_time=2.0)`; for a slow clock try `Game(ai_depth=10, ai_time=8.0)`. No code change is required — just edit the call below.

In [ ]:
Game(ai_depth=8, ai_time=4.0).run()
